# Inspect saved models


In [ ]:
from pathlib import Path
import math

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import your


## Set model paths


In [ ]:
import runpy

config_path = Path.cwd() / 'config.py'
if not config_path.is_file():
    config_path = Path.cwd().parent / 'config.py'
config = runpy.run_path(str(config_path))
filterbank_file_path = config['filterbank_file_path']
run_dir = config['run_dir']
cnn_checkpoint_path = run_dir / 'best_model_by_acc.pt'
statistical_model_path = run_dir / 'best_statistical_model.joblib'

for input_path in (filterbank_file_path, cnn_checkpoint_path, statistical_model_path):
    if not input_path.is_file():
        raise FileNotFoundError(f'Input file not found: {input_path}')


## Load saved models


In [ ]:
class CNN1D_RFI_256_logits(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7)
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=7)
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=10)
        self.pool3 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(6400, 256)
        self.fc2 = nn.Linear(256, 1)

    def forward(self, inputs):
        outputs = self.pool1(F.relu(self.conv1(inputs)))
        outputs = F.relu(self.conv2(outputs))
        outputs = F.pad(outputs, (0, 1))
        outputs = self.pool2(outputs)
        outputs = self.pool3(F.relu(self.conv3(outputs)))
        outputs = self.flatten(outputs)
        outputs = self.dropout(F.relu(self.fc1(outputs)))
        return self.fc2(outputs)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cnn_checkpoint = torch.load(cnn_checkpoint_path, map_location=device)
required_checkpoint_keys = {
    'epoch', 'model_state_dict', 'optimizer_state_dict',
    'validation_loss', 'validation_accuracy'
}
if set(cnn_checkpoint) != required_checkpoint_keys:
    raise ValueError('CNN checkpoint has an unexpected schema')

cnn_model = CNN1D_RFI_256_logits().to(device)
cnn_model.load_state_dict(cnn_checkpoint['model_state_dict'])
cnn_model.eval()

statistical_model = joblib.load(statistical_model_path)
required_bundle_keys = {
    'model_name', 'feature_columns', 'threshold', 'pipeline',
    'validation_metrics', 'test_metrics'
}
if set(statistical_model) != required_bundle_keys:
    raise ValueError('Statistical-model bundle has an unexpected schema')
if len(statistical_model['feature_columns']) != 3:
    raise ValueError('Statistical-model bundle must use three features')

print('CNN device:', device.type)
print('CNN checkpoint epoch:', cnn_checkpoint['epoch'])
print('Statistical model:', statistical_model['model_name'])
print('Statistical features:', statistical_model['feature_columns'])


## Define data functions


In [ ]:
n_time_samples = 256
filterbank_file = your.Your(str(filterbank_file_path))
n_segments = filterbank_file.your_header.nspectra // n_time_samples


def load_segment(segment_index):
    if not isinstance(segment_index, int) or not 0 <= segment_index < n_segments:
        raise ValueError(f'Segment index must be in [0, {n_segments - 1}]')
    segment = filterbank_file.get_data(
        nstart=segment_index * n_time_samples,
        nsamp=n_time_samples
    ).T
    expected_shape = (filterbank_file.your_header.nchans, n_time_samples)
    if segment.shape != expected_shape:
        raise ValueError('Filterbank segment has an unexpected shape')
    return segment


def compute_stats(values, axis=None):
    values = np.asarray(values)
    mean = values.mean(axis=axis, keepdims=True)
    std = values.std(axis=axis, keepdims=True)
    centered = values - mean
    std_safe = np.where(std == 0, 1.0, std)
    skew = (centered ** 3).mean(axis=axis, keepdims=True) / (std_safe ** 3)
    kurt = (centered ** 4).mean(axis=axis, keepdims=True) / (std_safe ** 4) - 3.0
    return mean.squeeze(), std.squeeze(), skew.squeeze(), kurt.squeeze()


def normalize_per_channel(segment):
    channel_min = segment.min(axis=1, keepdims=True)
    channel_max = segment.max(axis=1, keepdims=True)
    channel_range = channel_max - channel_min
    channel_range[channel_range == 0] = 1.0
    return 256.0 * (segment - channel_min) / channel_range


def compute_ratio(channel_values, segment_value):
    if segment_value == 0:
        return np.full_like(channel_values, np.nan, dtype=float)
    return np.asarray(channel_values, dtype=float) / segment_value


def segment_features(segment):
    mean_o_segment, std_o_segment, skew_o_segment, kurt_o_segment = compute_stats(segment)
    normalized_segment = normalize_per_channel(segment)
    mean_n_segment, std_n_segment, skew_n_segment, kurt_n_segment = compute_stats(normalized_segment)

    mean_o, std_o, skew_o, kurt_o = compute_stats(segment, axis=1)
    mean_n, std_n, skew_n, kurt_n = compute_stats(normalized_segment, axis=1)

    return pd.DataFrame({
        'mean_o': mean_o, 'std_o': std_o, 'skew_o': skew_o, 'kurt_o': kurt_o,
        'mean_n': mean_n, 'std_n': std_n, 'skew_n': skew_n, 'kurt_n': kurt_n,
        'mean_o_ratio': compute_ratio(mean_o, mean_o_segment),
        'std_o_ratio': compute_ratio(std_o, std_o_segment),
        'skew_o_ratio': compute_ratio(skew_o, skew_o_segment),
        'kurt_o_ratio': compute_ratio(kurt_o, kurt_o_segment),
        'mean_n_ratio': compute_ratio(mean_n, mean_n_segment),
        'std_n_ratio': compute_ratio(std_n, std_n_segment),
        'skew_n_ratio': compute_ratio(skew_n, skew_n_segment),
        'kurt_n_ratio': compute_ratio(kurt_n, kurt_n_segment)
    })


## Define model predictors


In [ ]:
def predict_cnn_channels(model, segment):
    # Keep the CNN input normalization identical to training.
    channels = np.asarray(segment, dtype=np.float32)
    channel_maxima = np.max(np.abs(channels), axis=1, keepdims=True)
    channel_maxima[channel_maxima < 1e-8] = 1.0
    channels = channels / channel_maxima
    inputs = torch.from_numpy(channels).unsqueeze(1).to(device)
    with torch.no_grad():
        scores = torch.sigmoid(model(inputs)).squeeze(1).cpu().numpy()
    return scores, scores >= 0.5


def predict_statistical_channels(model, segment):
    values = segment_features(segment).loc[:, model['feature_columns']]
    pipeline = model['pipeline']
    if hasattr(pipeline, 'predict_proba'):
        scores = pipeline.predict_proba(values)[:, 1]
    else:
        scores = 1.0 / (1.0 + np.exp(-pipeline.decision_function(values)))
    return scores, scores >= model['threshold']


## Define inspection functions


In [ ]:
def contiguous_ranges(mask):
    changes = np.diff(np.concatenate([[False], mask, [False]]).astype(np.int8))
    return zip(np.flatnonzero(changes == 1), np.flatnonzero(changes == -1))


def inspect_indices(model, predictor, segment_indices):
    inspections = []
    for segment_index in segment_indices:
        segment = load_segment(segment_index)
        scores, mask = predictor(model, segment)
        if scores.shape != (segment.shape[0],) or mask.shape != (segment.shape[0],):
            raise ValueError('Model prediction has an unexpected shape')
        inspections.append({
            'segment_index': segment_index,
            'segment': segment,
            'scores': scores,
            'mask': mask
        })
    return inspections


def palette_shape(n_segments):
    n_columns = math.ceil(math.sqrt(n_segments))
    return math.ceil(n_segments / n_columns), n_columns


def draw_inspection_pairs(inspections, model_label):
    n_rows, n_columns = palette_shape(len(inspections))
    figure, axes = plt.subplots(n_rows, 2 * n_columns, figsize=(7 * n_columns, 3 * n_rows), squeeze=False)
    masked_cmap = plt.get_cmap('gray').copy()
    masked_cmap.set_bad(color='white')

    for pair_index, inspection in enumerate(inspections):
        row_index, column_index = divmod(pair_index, n_columns)
        original_axis = axes[row_index, 2 * column_index]
        masked_axis = axes[row_index, 2 * column_index + 1]
        segment = inspection['segment']
        minimum = float(np.min(segment))
        maximum = float(np.max(segment))
        original_axis.imshow(segment, aspect='auto', origin='lower', cmap='gray', vmin=minimum, vmax=maximum)
        original_axis.set_title(f"Segment {inspection['segment_index']}", fontsize=9)

        masked_image = segment.astype(float).copy()
        masked_image[inspection['mask']] = np.nan
        masked_axis.imshow(masked_image, aspect='auto', origin='lower', cmap=masked_cmap, vmin=minimum, vmax=maximum)
        for start, stop in contiguous_ranges(inspection['mask']):
            masked_axis.axhspan(start - 0.5, stop - 0.5, color='red', alpha=0.35)
        masked_axis.set_title(f"NBRFI: {inspection['mask'].sum()}", fontsize=9)

        for axis in (original_axis, masked_axis):
            axis.set_xticks([])
            axis.set_yticks([])

    for pair_index in range(len(inspections), n_rows * n_columns):
        row_index, column_index = divmod(pair_index, n_columns)
        axes[row_index, 2 * column_index].set_visible(False)
        axes[row_index, 2 * column_index + 1].set_visible(False)

    figure.suptitle(model_label)
    figure.tight_layout()
    plt.show()


def inspect_one_segment(model, predictor, model_label, segment_index):
    inspections = inspect_indices(model, predictor, [segment_index])
    draw_inspection_pairs(inspections, model_label)


def inspect_random_palette(model, predictor, model_label, n_palette_segments, random_state):
    if not isinstance(n_palette_segments, int) or not 1 <= n_palette_segments <= n_segments:
        raise ValueError(f'Palette size must be in [1, {n_segments}]')
    random_generator = np.random.default_rng(random_state)
    segment_indices = random_generator.choice(n_segments, size=n_palette_segments, replace=False).tolist()
    inspections = inspect_indices(model, predictor, segment_indices)
    draw_inspection_pairs(inspections, model_label)
    return segment_indices


def inspect_index_palette(model, predictor, model_label, segment_indices):
    if not segment_indices:
        raise ValueError('Set at least one segment index')
    inspections = inspect_indices(model, predictor, segment_indices)
    draw_inspection_pairs(inspections, model_label)


## Inspect one segment


In [ ]:
single_segment_index = None

if not isinstance(single_segment_index, int):
    raise ValueError('Set single_segment_index to a segment index')

inspect_one_segment(cnn_model, predict_cnn_channels, 'CNN', single_segment_index)
inspect_one_segment(
    statistical_model,
    predict_statistical_channels,
    f"Statistical {statistical_model['model_name']}",
    single_segment_index
)


## Inspect a random palette


In [ ]:
random_palette_size = 25
random_palette_seed = 42

random_palette_indices = inspect_random_palette(
    cnn_model,
    predict_cnn_channels,
    'CNN',
    random_palette_size,
    random_palette_seed
)
inspect_index_palette(
    statistical_model,
    predict_statistical_channels,
    f"Statistical {statistical_model['model_name']}",
    random_palette_indices
)


## Inspect a specified palette


In [ ]:
specified_segment_indices = []

if not specified_segment_indices:
    raise ValueError('Set specified_segment_indices to a non-empty list')

# Use the same indices for direct model comparison.
inspect_index_palette(
    cnn_model,
    predict_cnn_channels,
    'CNN',
    specified_segment_indices
)
inspect_index_palette(
    statistical_model,
    predict_statistical_channels,
    f"Statistical {statistical_model['model_name']}",
    specified_segment_indices
)
